# Lecture 3: Distributed Paradigms

## 1. Hardware Primitives: PCIe, NVLink, and Ethernet
Why does Spark fail at parameter aggregation for massive deep learning models?
In standard **Ethernet** architectures (like Hadoop/Spark clusters), communication relies on standard hierarchical network switches over TCP/IP. The data path requires moving memory from the GPU -> PCIe bus -> CPU -> NIC -> Switch -> NIC -> CPU -> PCIe bus -> GPU. This CPU intervention causes catastrophic overhead.

Modern AI topologies (like NVIDIA DGX SuperPODs) leverage **InfiniBand / NVLink** to establish Remote Direct Memory Access (RDMA). A GPU can directly write into the VRAM of a remote GPU without waking up the host CPU, allowing bandwidth scaling into terabytes per second.


## 2. Distributed Strategies: MapReduce vs Collective Communication

**The Parameter Server (Spark-Style Hub-and-Spoke):**
Let $M$ be model size in bytes, $N$ be nodes, and $B$ be network bandwidth. 
The central parameter server must receive gradients from all nodes, totalizing $M \times N$ bytes. Because all traffic terminates at a single Network Interface Card, it becomes a hard limit:
$$T_{PS} = \frac{M \cdot N}{B}$$

**Ring-AllReduce (Collective Communication):**
Nodes form a logical ring. Instead of sending the full model to a hub, nodes segment the model $M$ into $N$ chunks.
- **Scatter-reduce:** Data flows in a ring for $N-1$ steps. In step $i$, each node sends an $\frac{M}{N}$ chunk to its neighbor. Gradients accumulate incrementally in the loop.
- **All-gather:** The sequence iterates exactly $N-1$ extra steps to distribute the completed sums back down the ring.

By ensuring all nodes transmit and receive simultaneously constantly, total communication time is independent of scale $N$:
$$T_{Ring} = 2 \times \frac{M(N-1)}{N \cdot B}$$

Taking the limit $N \to \infty$:
$$T_{Ring} \approx \frac{2M}{B}$$
This algorithmic design proves horizontal scaling mathematically possible!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, VBox, HBox, Output
import ipywidgets as widgets
from IPython.display import display

def plot_advanced_communication(N_max=1000, M=10.0, B=100.0):
    # Logarithmic-style spacing for smoother curves over huge magnitudes
    N_vals = np.linspace(2, N_max, 500)
    
    # Time equations
    T_PS = (M * N_vals) / B
    T_Ring = 2 * (M * (N_vals - 1)) / (N_vals * B)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Subplot 1: Time Scaling
    ax1.plot(N_vals, T_PS, label='Parameter Server Time', color='#e74c3c', linewidth=2.5)
    ax1.plot(N_vals, T_Ring, label='Ring-AllReduce Time', color='#2ecc71', linewidth=2.5)
    
    ax1.set_ylim(0, max(2 * M / B * 3, 2)) # Box the graph specifically to highlight the Ring plateau
    ax1.set_xlim(2, N_max)
    ax1.set_xlabel('Number of Nodes ($N$)')
    ax1.set_ylabel('Algorithm Time (Seconds)')
    ax1.set_title(f'Scaling Time Limits (Size: {M} GB, Bandwidth: {B} GB/s)')
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.7)
    
    # Subplot 2: Bandwidth Efficiency per Node
    efficiency_ps = B / N_vals
    efficiency_ring = np.ones_like(N_vals) * B
    
    ax2.plot(N_vals, efficiency_ps, label='PS Avg Available Bandwidth', color='#e74c3c', linewidth=2.5)
    ax2.plot(N_vals, efficiency_ring, label='Ring Avg Available Bandwidth', color='#2ecc71', linewidth=2.5)
    
    ax2.set_xlim(2, N_max)
    ax2.set_xlabel('Number of Nodes ($N$)')
    ax2.set_ylabel('Effective Bandwidth (GB/s)')
    ax2.set_title('Network Topology Saturation')
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

# Set up interactive sliders
print("Interactive Hardware Simulator")
ui = widgets.interactive(
    plot_advanced_communication,
    N_max=widgets.IntSlider(min=2, max=30000, step=100, value=1000, description='Max Nodes:'),
    M=widgets.FloatSlider(min=0.1, max=100.0, step=0.1, value=10.0, description='Model (GB):'),
    B=widgets.FloatSlider(min=10.0, max=1000.0, step=10.0, value=100.0, description='BW (GB/s):')
)
display(ui)

### Exercise
Manipulate the sliders above to simulate a massive **30,000 GPU cluster**. Looking at the "Network Topology Saturation" graph, why does the Parameter Server effective bandwidth asymptote to zero?
Explain why AI hardware engineers enforce torus arrays and specialized NVLink rings over standard switch hubs:

**(Student explanation here)**